In [1]:
# ============================================================
# BirdCLEF 2026 | Single-flag blend inference
# Change only EXP_ID to switch blend strategy
# ============================================================

import os
import re
import time
import glob
import warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

START = time.time()
print(f"PyTorch {torch.__version__}")

# ══════════════════════════════════════════════════════════════
# CHANGE ONLY THIS FLAG
# 0: finetuned only
# 1: baseline only
# 2: prob  0.8 ft + 0.2 base
# 3: prob  0.7 ft + 0.3 base
# 4: prob  0.5 ft + 0.5 base
# 5: logit 0.8 ft + 0.2 base
# 6: logit 0.7 ft + 0.3 base
# 7: logit 0.5 ft + 0.5 base
# ══════════════════════════════════════════════════════════════
EXP_ID = 2

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

cfg = Config()

# ══════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════
DATA_ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")

FINETUNED_CKPT = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB872.pt"
BASELINE_CKPT  = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB862.pt"

assert os.path.exists(FINETUNED_CKPT), f"Missing finetuned checkpoint: {FINETUNED_CKPT}"
assert os.path.exists(BASELINE_CKPT), f"Missing baseline checkpoint: {BASELINE_CKPT}"

BLEND_OPTIONS = [
    ("finetuned_only",    {"mode": "prob",  "ft": 1.0, "base": 0.0}),  # 0
    ("baseline_only",     {"mode": "prob",  "ft": 0.0, "base": 1.0}),  # 1
    ("prob_ft80_base20",  {"mode": "prob",  "ft": 0.8, "base": 0.2}),  # 2
    ("prob_ft70_base30",  {"mode": "prob",  "ft": 0.7, "base": 0.3}),  # 3
    ("prob_ft50_base50",  {"mode": "prob",  "ft": 0.5, "base": 0.5}),  # 4
    ("logit_ft80_base20", {"mode": "logit", "ft": 0.8, "base": 0.2}),  # 5
    ("logit_ft70_base30", {"mode": "logit", "ft": 0.7, "base": 0.3}),  # 6
    ("logit_ft50_base50", {"mode": "logit", "ft": 0.5, "base": 0.5}),  # 7
]

assert 0 <= EXP_ID < len(BLEND_OPTIONS), f"EXP_ID must be in [0, {len(BLEND_OPTIONS)-1}]"
SELECTED_NAME, SELECTED_SPEC = BLEND_OPTIONS[EXP_ID]

print(f"Selected blend: EXP_ID={EXP_ID} -> {SELECTED_NAME}")

# Species list from submission columns
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)
print(f"Species: {len(SPECIES)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ══════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        x = x.pow(1.0 / p)
        return x

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)

        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)

        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        segmentwise_logit = cls.permute(0, 2, 1)

        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob": clipwise_prob,
            "segmentwise_logit": segmentwise_logit,
        }

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool="",
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)

# ══════════════════════════════════════════════════════════════
# MEL TRANSFORM
# ══════════════════════════════════════════════════════════════
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)

        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)

        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)

        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)

        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)

        mel = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        return mel

# ══════════════════════════════════════════════════════════════
# CHECKPOINT HELPERS
# ══════════════════════════════════════════════════════════════
def safe_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def extract_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt

def extract_meta(ckpt):
    if isinstance(ckpt, dict):
        epoch = ckpt.get("epoch", "?")
        stage = ckpt.get("stage", "?")
        metrics = ckpt.get("metrics", {})
        auc = metrics.get("macro_auc", "?") if isinstance(metrics, dict) else "?"
        return epoch, stage, auc
    return "?", "raw_state_dict", "?"

def load_model(ckpt_path, tag):
    ckpt = safe_load_checkpoint(ckpt_path)
    model = SEDModel(cfg)
    model.load_state_dict(extract_state_dict(ckpt), strict=True)
    model.to(device).eval()
    epoch, stage, auc = extract_meta(ckpt)
    print(f"{tag}: loaded | epoch={epoch} | stage={stage} | val_auc={auc}")
    return model

# ══════════════════════════════════════════════════════════════
# LOAD MODELS
# ══════════════════════════════════════════════════════════════
baseline_model = load_model(BASELINE_CKPT, "baseline")
finetuned_model = load_model(FINETUNED_CKPT, "finetuned")
mel_transform = MelSpectrogramTransform(cfg).to(device).eval()

# ══════════════════════════════════════════════════════════════
# TEST FILE DISCOVERY
# ══════════════════════════════════════════════════════════════
row_pattern = re.compile(r"^(.*)_(\d+)$")

def parse_row_id(row_id: str):
    m = row_pattern.match(str(row_id))
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)

for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
use_fallback = len(test_files) == 0

if use_fallback:
    fallback_files = []
    for stem in sorted(expected_by_stem.keys()):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p):
            fallback_files.append(p)
    test_files = fallback_files
    print(f"No test .ogg files found; using aligned local fallback with {len(test_files)} soundscape files.")
else:
    print(f"Found {len(test_files)} test soundscape files.")

# ══════════════════════════════════════════════════════════════
# AUDIO LOADING
# ══════════════════════════════════════════════════════════════
def load_soundscape(path):
    stem = Path(path).stem
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return y, stem

print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as executor:
    results = list(executor.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time() - t0:.1f}s")

# ══════════════════════════════════════════════════════════════
# BLEND HELPER
# ══════════════════════════════════════════════════════════════
def probs_from_blend(base_logits, ft_logits, base_probs, ft_probs, mode, ft_w, base_w):
    if mode == "prob":
        probs = ft_w * ft_probs + base_w * base_probs
    elif mode == "logit":
        logits = ft_w * ft_logits + base_w * base_logits
        probs = torch.sigmoid(logits)
    else:
        raise ValueError(f"Unknown blend mode: {mode}")

    probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    probs = torch.clamp(probs, 0.0, 1.0)
    return probs

# ══════════════════════════════════════════════════════════════
# INFERENCE
# ══════════════════════════════════════════════════════════════
CHUNK = cfg.chunk_samples
all_row_ids = []
all_preds = []

print("Running inference...")
t0 = time.time()

with torch.no_grad():
    for audio, stem in results:
        if stem in expected_by_stem:
            target_end_secs = expected_by_stem[stem]
            n_chunks = max(target_end_secs) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // CHUNK)

        padded_len = n_chunks * CHUNK
        if len(audio) < padded_len:
            audio = np.pad(audio, (0, padded_len - len(audio)))
        else:
            audio = audio[:padded_len]

        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak

        chunks = audio.reshape(n_chunks, CHUNK)
        chunks_tensor = torch.from_numpy(chunks).float().to(device)

        mel = mel_transform(chunks_tensor)

        out_base = baseline_model(mel)
        out_ft = finetuned_model(mel)

        base_logits = torch.nan_to_num(out_base["clipwise_logit"], nan=0.0, posinf=20.0, neginf=-20.0)
        ft_logits   = torch.nan_to_num(out_ft["clipwise_logit"],   nan=0.0, posinf=20.0, neginf=-20.0)

        base_probs = torch.nan_to_num(out_base["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)
        ft_probs   = torch.nan_to_num(out_ft["clipwise_prob"],   nan=0.0, posinf=1.0, neginf=0.0)

        if stem in expected_by_stem:
            valid_indices = [(end_sec // int(cfg.chunk_duration)) - 1 for end_sec in expected_by_stem[stem]]
            valid_indices = [i for i in valid_indices if 0 <= i < n_chunks]
            target_row_ids = [f"{stem}_{(i + 1) * int(cfg.chunk_duration)}" for i in valid_indices]
        else:
            valid_indices = list(range(n_chunks))
            target_row_ids = [f"{stem}_{(i + 1) * int(cfg.chunk_duration)}" for i in valid_indices]

        probs = probs_from_blend(
            base_logits=base_logits,
            ft_logits=ft_logits,
            base_probs=base_probs,
            ft_probs=ft_probs,
            mode=SELECTED_SPEC["mode"],
            ft_w=SELECTED_SPEC["ft"],
            base_w=SELECTED_SPEC["base"],
        ).detach().cpu().numpy()

        all_row_ids.extend(target_row_ids)
        all_preds.extend(probs[valid_indices])

elapsed = time.time() - t0
print(f"Inference done in {elapsed:.1f}s")

# ══════════════════════════════════════════════════════════════
# BUILD SUBMISSION
# ══════════════════════════════════════════════════════════════
def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name="row_id"),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name="row_id"),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

    missing_ids = [rid for rid in expected_ids if rid not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name="row_id"),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index()

submission = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv("submission.csv", index=False)
submission.to_csv(f"submission_{SELECTED_NAME}.csv", index=False)

# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
print(f"\nSaved: submission.csv")
print(f"Saved: submission_{SELECTED_NAME}.csv")
print(f"Blend used: {SELECTED_NAME}")
print(f"submission shape: {submission.shape}")
print(f"Total time: {time.time() - START:.1f}s ({(time.time() - START)/60:.1f} min)")
print(submission.head())

PyTorch 2.9.0+cpu
Selected blend: EXP_ID=2 -> prob_ft80_base20
Species: 234
Device: cpu
baseline: loaded | epoch=13 | stage=? | val_auc=0.9478199963260324
finetuned: loaded | epoch=3 | stage=stage1 | val_auc=0.9959889456149783
No test .ogg files found; using aligned local fallback with 0 soundscape files.
Loading audio...
Loaded 0 files in 0.0s
Running inference...
Inference done in 0.0s

Saved: submission.csv
Saved: submission_prob_ft80_base20.csv
Blend used: prob_ft80_base20
submission shape: (3, 235)
Total time: 2.0s (0.0 min)
                                    row_id  1161364  116570  1176823  1491113  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.0     0.0      0.0      0.0   
1  BC2026_Test_0001_S05_20250227_010002_10      0.0     0.0      0.0      0.0   
2  BC2026_Test_0001_S05_20250227_010002_15      0.0     0.0      0.0      0.0   

   1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  whwpic1  \
0      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0    